# Genome-wide annotation density tracks

Per-chromosome gene-density tracks (1 Mb windows, % of bases covered) for the
Liftoff transferred and Braker3 de-novo annotations, plus RepeatMasker repeat
density. Produces both the raw and rolling-smoothed density figures.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
from natsort import natsorted
import matplotlib.gridspec as gridspec


In [ ]:
chrom_length = pd.read_csv(
    f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.fasta.fai",
    sep="\t", names=["Name", "Length"], usecols=[0, 1])


In [ ]:
# Liftoff transferred annotation (gene features)
feat_lift = pd.read_csv(
    f"{PROJ_ROOT}/output/outputs-from-liftoff/hifiasm-041425-scaffolded-chrAssigned/hifiasm-041425-scaffolded-chrAssigned.gff",
    sep="\t", header=None,
    names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"],
    comment="#")
feat_lift = feat_lift[feat_lift["type"] == "gene"]


In [ ]:
# Braker3 de-novo annotation (gene features)
feat_braker = pd.read_csv(
    f"{PROJ_ROOT}/code/command-line-script/genome-annotation/annotate-braker3-results/annotate-gff/braker.uniprotBlast.interproscan.gff/braker.gff3",
    sep="\t", header=None,
    names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"],
    comment="#")
feat_braker = feat_braker[feat_braker["type"] == "gene"]


In [ ]:
# RepeatMasker repeats (all feature types)
df_repeat = pd.read_csv(
    f"{PROJ_ROOT}/figure/circos-plot/feature-overview/assembly_final.sorted.headerRenamed.fasta.out.chr.gff",
    sep="\t", header=None,
    names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"],
    comment="#")


In [ ]:
import pandas as pd
import numpy as np
from itertools import chain

def collapse_overlapping_features(df_gff):
    """
    Collapse overlapping features by merging them into contiguous intervals.
    
    Args:
        df_gff: DataFrame with columns ['chrom', 'start', 'end', ...]
    
    Returns:
        DataFrame with non-overlapping merged features
    """
    collapsed_features = []
    
    for chrom in df_gff['chrom'].unique():
        chrom_df = df_gff[df_gff['chrom'] == chrom].copy()
        chrom_df = chrom_df.sort_values('start')
        
        # Initialize with first feature
        current_start = chrom_df.iloc[0]['start']
        current_end = chrom_df.iloc[0]['end']
        
        for _, row in chrom_df.iloc[1:].iterrows():
            if row['start'] <= current_end:  # Overlapping or adjacent
                current_end = max(current_end, row['end'])
            else:
                # Save current merged feature and start new one
                collapsed_features.append({
                    'chrom': chrom,
                    'start': current_start,
                    'end': current_end
                })
                current_start = row['start']
                current_end = row['end']
        
        # Add the last merged feature
        collapsed_features.append({
            'chrom': chrom,
            'start': current_start,
            'end': current_end
        })
    
    return pd.DataFrame(collapsed_features)

def calculate_basepair_density_from_gff(df_gff, chr_sizes, window_size=1_000_000):
    """
    Calculate feature density as percentage of basepairs covered in each window.
    
    Args:
        df_gff: DataFrame with genomic features (columns: chrom, start, end)
        chr_sizes: DataFrame with chromosome sizes (columns: Name, Length)
        window_size: Size of sliding window in basepairs
    
    Returns:
        DataFrame with density calculated as percentage of bases covered
    """
    # First collapse overlapping features
    df_collapsed = collapse_overlapping_features(df_gff)
    
    # Convert chr_sizes to dictionary if it's a DataFrame
    if isinstance(chr_sizes, pd.DataFrame):
        chr_sizes_dict = dict(zip(chr_sizes['Name'], chr_sizes['Length']))
    else:
        chr_sizes_dict = chr_sizes
    
    density_data = []
    
    for chrom, length in chr_sizes_dict.items():
        # Get features for this chromosome
        chrom_features = df_collapsed[df_collapsed['chrom'] == chrom]
        
        # Create windows
        bins = np.arange(0, length, window_size)
        
        for start in bins:
            end = min(start + window_size, length)
            window_length = end - start
            
            # Find features that overlap with this window
            overlapping_features = chrom_features[
                (chrom_features['end'] > start) & 
                (chrom_features['start'] < end)
            ].copy()
            
            if overlapping_features.empty:
                coverage_bp = 0
            else:
                # Calculate total basepairs covered in this window
                overlapping_features['window_start'] = overlapping_features['start'].clip(lower=start)
                overlapping_features['window_end'] = overlapping_features['end'].clip(upper=end)
                overlapping_features['overlap_length'] = overlapping_features['window_end'] - overlapping_features['window_start']
                
                coverage_bp = overlapping_features['overlap_length'].sum()
            
            # Calculate density as percentage of window covered
            density_pct = (coverage_bp / window_length) * 100 if window_length > 0 else 0
            
            density_data.append([chrom, start, end, coverage_bp, density_pct])
    
    df_density = pd.DataFrame(density_data, 
                             columns=["contig", "start", "end", "covered_bp", "density_pct"])
    return df_density

# Example usage:
# df_density = calculate_basepair_density_from_gff(df_gff, chrom_length, window_size=1000000)

In [ ]:
df_braker_density = calculate_basepair_density_from_gff(feat_braker, chrom_length, window_size=1000000)
df_braker_density=df_braker_density[df_braker_density["contig"].str.contains("chr")]

In [ ]:
df_lift_density = calculate_basepair_density_from_gff(feat_lift, chrom_length, window_size=1000000)
df_lift_density=df_lift_density[df_lift_density["contig"].str.contains("chr")]

In [ ]:
df_repeat_density = calculate_basepair_density_from_gff(df_repeat, chrom_length, window_size=1000000)
df_repeat_density=df_repeat_density[df_repeat_density["contig"].str.contains("chr")]

In [ ]:
## Just look at the first 30 chromosome
chrom_length=chrom_length[chrom_length["Name"].str.contains("chr")]

In [ ]:
def plot_density_multiple_datasets(chromosome_lengths, density_datasets,  variable_to_plot, size,contigs_per_row=30):
        # Calculate genomic metrics
    chr_lengths = chromosome_lengths.groupby("Name")["Length"].max()
    contigs = natsorted(chr_lengths.index.tolist())
    chr_lengths = chr_lengths[contigs]
    n_contigs = len(contigs)
    
    # Normalize widths
    max_length = chr_lengths.max()
    relative_widths = chr_lengths / max_length
    
    # Calculate median coverage for each read type
    for dataset in density_datasets:
        dataset['ymax'] = dataset['df'][variable_to_plot].max()
    
    
    # Layout parameters
    n_rows = math.ceil(n_contigs / contigs_per_row) * len(datasets)
    
    # Create figure
    fig = plt.figure(figsize=(15, n_rows * 1.2))
        # ADD COMMON Y-AXIS LABEL USING fig.text()
    fig.text(-0.02, 0.5, "Annotation density", va='center', rotation='vertical', fontsize=size+2)
    fig.text(0.5, -0.17, "Chromosomes", va='center', rotation='horizontal', fontsize=size+2)
    gs = gridspec.GridSpec(n_rows, contigs_per_row,
                          width_ratios=relative_widths[:contigs_per_row],
                          hspace=0.3, wspace=0.1)
    # Plotting loop
    plot_counter = 0
    for contig_idx, contig in enumerate(contigs):
        for dataset_idx, dataset in enumerate(datasets):
            row = (contig_idx // contigs_per_row) * len(datasets) + dataset_idx
            col = contig_idx % contigs_per_row
            
            ax = plt.subplot(gs[row, col])
            plot_counter += 1
            
            # Plot data
            contig_data = dataset['df'][dataset['df']['contig'] == contig]
            midpoints = (contig_data['start'] + contig_data['end']) / 2
            ax.plot(midpoints, contig_data[variable_to_plot], color=dataset['color'], linewidth=0.8)
            
            # Set axes limits
            ax.set_ylim(0, dataset['ymax'])
            ax.set_xlim(0, chr_lengths[contig])
            
            # Configure ticks
            max_mb = chr_lengths[contig]/1e6
            major_ticks = np.arange(0, max_mb+ 1e-9, 50) * 1e6
            minor_ticks = np.arange(0, max_mb+ 1e-9, 10) * 1e6
            
            ax.set_xticks(major_ticks)
            ax.set_xticks(minor_ticks, minor=True)
            ax.set_xlim(0, chr_lengths[contig])
            
            # Smart formatting
            is_first_row_in_group = (row % len(datasets)) == 0
            is_last_contig_group = (contig_idx // contigs_per_row) == (math.ceil(n_contigs / contigs_per_row)) - 1
            is_first_col = col == 0
            
            # Title only on first row of each contig group
            if is_first_row_in_group:
                ax.set_title(contig, fontsize=size-3, rotation=45, pad=3)
            
            # X-axis labels only on bottom row
            if is_last_contig_group and (dataset_idx == len(datasets)-1):
                ax.set_xticklabels([f"{x/1e6:.0f}Mb" if (x in major_ticks and x != 0) else "" for x in major_ticks], fontsize=size-5,rotation=45)
                # ax.set_xlabel(f"{chr_lengths[contig]/1e6:.1f}Mb", fontsize=size-3, rotation=90)
            else:
                # ax.set_xticklabels([])
                # ax.set_xticks(minor_ticks, minor=True)
                ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
                # ax.set_xlim(0, chr_lengths[contig])
            
            # Y-axis labels only on first column
            if is_first_col:
                ax.set_ylabel(dataset['name'], fontsize=size)
                ax.tick_params(axis='y', labelsize=size)
            else:
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
            
            # Grid lines
            ax.grid(True, which='major', alpha=0.2)
            ax.grid(True, which='minor', alpha=0.1)
    
    # Hide unused axes
    for i in range(plot_counter, n_rows * contigs_per_row):
        row = i // contigs_per_row
        col = i % contigs_per_row
        fig.delaxes(gs[row, col])
    
    plt.subplots_adjust(left=0.06, right=0.98, bottom=0.01, top=0.95,hspace=0.01)
    return fig

In [ ]:
# Define datasets
datasets = [
    {'df': df_lift_density, 'color': "limegreen", 'name': 'LiftOff\ntransferred'},
    {'df': df_braker_density, 'color': 'green', 'name': 'Braker3\ndenovo'}, 
    {'df': df_repeat_density, 'color': 'grey', 'name': 'RepeatMasker\nrepeats'}
]

In [ ]:
plot=plot_density_multiple_datasets(chrom_length, datasets, "density_pct")
# plot.savefig('density_count_annotation.png', dpi=600,bbox_inches="tight")  # Instead of show()

In [ ]:
### Rolling window of 1 mb
# Define your window size (in bases)
window_size = 150000  # Match your bin size for natural smoothing
min_periods = 1      # Minimum points needed for calculation

for dataset in datasets:
    # Sort by position first (critical for rolling)
    dataset['df'] = dataset['df'].sort_values(['contig', 'start'])
    
    # Calculate rolling median coverage per chromosome
    dataset['df']['smoothed_density_pct'] = (dataset['df']
        .groupby('contig')['density_pct']
        .transform(lambda x: x.rolling(window=window_size//15000,  # Number of bins
                                 min_periods=min_periods,
                                 center=True).mean())
    )
    
    # Fill NA values at chromosome ends
    dataset['df']['smoothed_density_pct'] = dataset['df']['smoothed_density_pct'].fillna(
        dataset['df']['density_pct']
    )

In [ ]:
plot=plot_density_multiple_datasets(chrom_length, datasets, "smoothed_density_pct",14)
plot.savefig('density_count_annotation_rolling.png', dpi=600,bbox_inches="tight")  # Instead of show()